# build_dataset — labeled (topic, doc) pairs + splits on the frozen representation

Emits training/val **pairs** (`{source, topic_id, topic_text, doc_id, label}`) from the judged qrels —
**no doc text baked in.** The document representation is applied at tokenize time from `ExperimentConfig`
(frozen `elig_first-L512`), so it can never drift from what the model is evaluated on (the §2g bug: the old
`train_clf_data.jsonl` baked in eligibility-only `ctmatch_ir` text). Test = TREC22, held out.


## Setup (Colab)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q datasets transformers pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from datasets import load_dataset
from ctmatch.experiments import ExperimentConfig, clf_pairs
cfg = ExperimentConfig(data_root=DATA_ROOT)
print('repr:', cfg.repr_tag())


In [ ]:
# Corpus id set — keep only judged docs that actually exist in the corpus (join happens at train time).
idx = load_dataset(cfg.index2docid_hf, data_files='index2docid.txt', split='train')
corpus_ids = {r['text'].strip() for r in idx}
print(f'{len(corpus_ids):,} corpus docs')


In [ ]:
# All judged pairs for the TRAIN sources (TREC21 + KZ). TREC22 is held out (never here).
TRAIN_SETS = ['trec21', 'kz']
pairs = clf_pairs(cfg, TRAIN_SETS, id_filter=corpus_ids)
df = pd.DataFrame(pairs)
print(f'{len(df):,} pairs | {df.topic_id.nunique()} topics')
print(df.groupby('source').label.value_counts().unstack(fill_value=0))


In [ ]:
# Topic-level train/val split (never split within a topic — avoids leakage).
VAL_FRAC = 0.15; SEED = 42
topics = sorted(df.topic_id.unique())
rng = np.random.default_rng(SEED); rng.shuffle(topics)
n_val = max(1, int(len(topics) * VAL_FRAC))
val_topics = set(topics[:n_val]); train_topics = set(topics[n_val:])
train_df = df[df.topic_id.isin(train_topics)]; val_df = df[df.topic_id.isin(val_topics)]
print(f'train: {len(train_df):,} pairs / {len(train_topics)} topics | val: {len(val_df):,} / {len(val_topics)}')


In [ ]:
# Write pairs + the split manifest. Tagged with repr so a consumer knows the intended representation.
os.makedirs(cfg.path('data'), exist_ok=True)
def dump(frame, path):
    with open(path, 'w') as f:
        for r in frame.to_dict('records'): f.write(json.dumps(r) + '\n')
dump(train_df, cfg.path('data/clf_pairs_train.jsonl'))
dump(val_df,   cfg.path('data/clf_pairs_val.jsonl'))
json.dump({'repr_tag': cfg.repr_tag(), 'train_sets': TRAIN_SETS, 'test_set': 'trec22',
           'val_frac': VAL_FRAC, 'seed': SEED,
           'train_topics': sorted(train_topics), 'val_topics': sorted(val_topics)},
          open(cfg.path('data/clf_splits.json'), 'w'), indent=1)
print('wrote clf_pairs_train.jsonl, clf_pairs_val.jsonl, clf_splits.json')
